TO ADD: se i punti sono corrispondenti => vale la relazione della matrice fondamentale, ma non è detto il viceversa, mettiamolo come domanda (infinit punti giacciono sulla retta epipolare Fx, ma solo x' è corrispondente.)

TO ADD: qui mettiamo le immagini reali, con un punto ad esempio X_1 segnato in entrambe le immagini, per le immagini usiamo x_i e x_i' minuscolo, sempre con la convenzione matrici \mathsf e punti vettori \mathbf.
# The fundamental matrix

This notebook is about the constraint that ties two views of the same scene: epipolar geometry, and the object that carries it when the cameras are uncalibrated — the fundamental matrix.

We start from a question. Given a point 
𝑥
x in the first image and a point 
𝑥
′
x
′
 in the second, how do we decide whether they are images of the same 3D point?

The question, to some extent, can be answered from images alone, without requiring as though it needs a reconstruction: find the world point, project it back, compare. It does not. There is a condition that involves only the two image points and the two cameras, it is linear in each of them, and it reads

𝑥
′
⊤
𝐹
 
𝑥
=
0.
x
′⊤
Fx=0.

 We characterise corresponding points algebraically, to realize that the images of corrisponding points must satisfy a bi-linear realtion. We then revisit this conditiont geometrically and present the main characters of epipolar geometry, including the fundamental matrix. Finally we estimate the funamental matrix from correspondences clicked by hand, where nothing is exact and the difficulties begin.

As usual, the object throughout is the Origami House. We start to work out the geometry, in a noisiless configuration,  before moving then to real data.


In [ ]:

#| echo: false
import sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules and not Path("cv-dojo").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/magrilu/cv-dojo.git"], check=True)
    import os
    os.chdir("cv-dojo")

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "cvdojo").exists():
        sys.path.insert(0, str(parent / "src")); break

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection

from cvdojo.house import (load_image, load_model, load_two_view_cameras,
                          load_annotation, project)
from cvdojo.plotting import (clip_line_to_image, draw_line_in_image,
                             pairwise_intersections, ACCENT)
from cvdojo.scene import skew, camera_center, set_axes_equal

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": False})

BLUE, GREY = "#3288BD", "0.45"

model = load_model()
V3 = {k: np.array(v, float) for k, v in model["vertices"].items()}
EDGES = model["edges"]
P, Pp = load_two_view_cameras()

I  = load_image("IMG_4331.jpeg")
Ip = load_image("IMG_4337.jpeg")
H_IMG, W_IMG = I.shape[:2]

IDS = list(V3)
X3 = np.array([V3[k] for k in IDS])
xs  = project(P,  X3)          # exact projections: our synthetic data
xps = project(Pp, X3)
h = lambda a: np.c_[np.atleast_2d(a), np.ones(len(np.atleast_2d(a)))]


## 1. When do two points correspond?

Here are the two views, drawn by projecting the model of the Origami House
through the two cameras $P$ and $P'$. For now, we are working with  synthetic images: every point sits exactly where the
geometry says it should.

Ten vertices are marked. The thin wireframe is there so that you can recognise
the house and see which corner is which.


In [ ]:
#| echo: false
fig = plt.figure(figsize=(16, 5.4))

# --- the object itself, in three dimensions -------------------------------
ax0 = fig.add_subplot(1, 3, 1, projection="3d")
for a, b in EDGES:
    ax0.plot(*zip(V3[a], V3[b]), color=GREY, lw=1.0)
ctr = np.mean(list(V3.values()), axis=0)
for k, Q in V3.items():
    ax0.scatter(*Q, s=18, color=ACCENT, depthshade=False, zorder=5)
    off = Q - ctr
    off[:2] = 0.55 * off[:2] / (np.linalg.norm(off[:2]) or 1)
    off[2] = 0.30 * np.sign(off[2])
    ax0.text(*(Q + off), rf"$\mathbf{{X}}_{{{k[1:]}}}$",
             color=ACCENT, fontsize=9, ha="center", va="center")
d = model["dimensions_cm"]
ax0.set_box_aspect((d["long_side"], d["depth"], d["total_height"]))
ax0.view_init(elev=18, azim=-62); ax0.set_axis_off()
ax0.set_title("the object\n$\\mathbf{X}_i \\in \\mathbb{P}^3$", fontsize=11)

# --- and its two images ---------------------------------------------------
for j, (pts, ttl, prime) in enumerate(
        [(xs, r"view 1:  $\mathbf{x}_i = \mathsf{P}\,\mathbf{X}_i$", False),
         (xps, r"view 2:  $\mathbf{x}'_i = \mathsf{P}'\mathbf{X}_i$", True)]):
    ax = fig.add_subplot(1, 3, j + 2)
    Q = {k: p for k, p in zip(IDS, pts)}
    for a, b in EDGES:
        ax.plot(*zip(Q[a], Q[b]), color=GREY, lw=0.8)
    ax.scatter(pts[:, 0], pts[:, 1], s=40, facecolors="none",
               edgecolors=ACCENT, lw=1.6, zorder=5)
    for k, p in Q.items():
        lab = (rf"$\mathbf{{x}}'_{{{k[1:]}}}$" if prime
               else rf"$\mathbf{{x}}_{{{k[1:]}}}$")
        ax.text(p[0] + 16, p[1] - 16, lab, color=BLUE, fontsize=9)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_title(ttl, fontsize=11)
    ax.set_xlabel("u  [px]"); ax.set_ylabel("v  [px]")
plt.tight_layout(); plt.show()


Now let's start by the definition of corresponding points.

::: {#def-correspondence}
## Correspondence

Two image points $\mathbf{x}$ and $\mathbf{x}'$ correspond if 
if there exists a world point $\mathbf{X}$ that projects to both:

$$\lambda\,\mathbf{x} = \mathsf{P}\mathbf{X}, \qquad
  \lambda'\mathbf{x}' = \mathsf{P}'\mathbf{X}.$$
:::

The scale factors $\lambda, \lambda'$ are there because a projective point is
only defined up to scale.

We can read the two equations as a single linear system in the unknowns
$(\mathbf{X}, \lambda, \lambda')$,  six unknowns, since $\mathbf{X}$ is a
homogeneous 4-vector:

$$
\underbrace{\begin{bmatrix}
\mathsf{P} & -\mathbf{x} & \mathbf{0} \\[2pt]
\mathsf{P}' & \mathbf{0} & -\mathbf{x}'
\end{bmatrix}}_{\textstyle \mathsf{L}}
\begin{bmatrix} \mathbf{X} \\ \lambda \\ \lambda' \end{bmatrix} = \mathbf{0}.
$$

$\mathsf{L}$ is $6 \times 6$ and the system is homogeneous, so it always has the
solution $\mathbf{0}$ — which is no solution at all, because $\mathbf{X} =
\mathbf{0}$ is not a point of $\mathbb{P}^3$. A genuine world point exists
exactly when the system has a **non-trivial** solution, that is, $\mathbf{x}$ and $\mathbf{x}'$ are corresponding points  if

$$\det \mathsf{L} = 0. {#eq-detL}$$

Notice what this expression contains: the two camera matrices, and the two image
points. No $\mathbf{X}$. Whether two points correspond can be settled without
ever computing the world point they would come from.

Let us watch it happen, considering for example $\mathbf{x}_9$ and $\mathbf{x}'_9$


In [ ]:
def L_matrix(P, Pp, x, xp):
    """The 6x6 system expressing that x and x' are images of a common point."""
    L = np.zeros((6, 6))
    L[:3, :4] = P;   L[:3, 4] = -x
    L[3:, :4] = Pp;  L[3:, 5] = -xp
    return L

k = "X9"                                  # the right end of the ridge
x  = h(xs[IDS.index(k)])[0]
xp = h(xps[IDS.index(k)])[0]

L = L_matrix(P, Pp, x, xp)
print(f"rank of L : {np.linalg.matrix_rank(L)}  (of 6)")
print(f"det L     : {np.linalg.det(L):.3e}")


Rank five, determinant zero: the null space is one-dimensional, and that single
direction is the world point. Let us extract it and see what it is.


In [ ]:
sol = np.linalg.svd(L)[2][-1]
X_rec, lam, lamp = sol[:4], sol[4], sol[5]
X_rec = X_rec[:3] / X_rec[3]

print("recovered X :", np.round(X_rec, 3), "cm")
print("model    X  :", np.round(V3[k], 3), "cm")
print("difference  :", f"{np.linalg.norm(X_rec - V3[k]):.2e} cm")


So $\mathbf{X}_9$ is indeed the soluiton of the system Now the same computation with a pair that does **not** correspond. We keep
$\mathbf{x}$ where it is, and take for $\mathbf{x}'$ the image of a different
vertex — the *left* end of the ridge instead of the right one. Geometrically
these two rays pass nowhere near each other.


In [ ]:
xp_wrong = h(xps[IDS.index("X8")])[0]        # image of a different vertex

L_bad = L_matrix(P, Pp, x, xp_wrong)
print(f"rank of L : {np.linalg.matrix_rank(L_bad)}  (of 6)")
print(f"det L     : {np.linalg.det(L_bad):.3e}")
print(f"ratio of the smallest to the largest singular value: "
      f"{np.linalg.svd(L_bad)[1][-1] / np.linalg.svd(L_bad)[1][0]:.2e}")


As expected we get full rank. The only solution is the zero vector, which is not a point in $\mathbb{P}^3$: the
system is inconsistent, and the two image points cannot be images of the same
corner of the house.

So we have a test: if two points correspond, then [-@eq-detL] holds.
Note that $\det\mathsf{L}$ is a number computed from four things — the two cameras $\mathsf P, \mathsf P'$ and the
two image points $\mathbf{x},\mathbf{x'}$. It does not involve the 3D point $\mathbf{X}$. The constraint has a nice property: $\mathbf{x}$ appears
in one column and nowhere else, and $\mathbf{x}'$ in another. A determinant is
linear in each of its columns. Therefore $\det\mathsf{L}$ is linear in
$\mathbf{x}$ and linear in $\mathbf{x}'$. It's a **bilinear form**. Every
bilinear form on $\mathbb{R}^3 \times \mathbb{R}^3$ can be written in exactly one
way as

$$\det \mathsf{L} \;=\; \mathbf{x}'^\top \mathsf{F}\, \mathbf{x}$$

for a single $3 \times 3$ matrix $\mathsf{F}$.  What remains is to find out what this matrix means.


## 2. The epipolar geometry


Go back to the definition of corresponding points and read it the other way round. The point
$\mathbf{x}$ in the first image  determines
a **ray**, the set of all world points that project there, running from the
camera centre $\mathsf{C}$ out through the image plane. The same is true of
$\mathbf{x}'$ and its ray from $\mathsf{C}'$.

Two points correspond exactly when their two rays meet.


In [ ]:
#| echo: false
def ray(P, x, tmin=-0.4, tmax=1.9, n=2):
    """Points along the optical ray of the image point x."""
    C = np.linalg.svd(P)[2][-1]; C = C[:3] / C[3]
    d = np.linalg.pinv(P) @ x; d = d[:3] / d[3] - C
    ts = np.linspace(tmin, tmax, n)
    return C, np.array([C + t * d for t in ts])

C  = np.linalg.svd(P)[2][-1];  C  = C[:3] / C[3]
Cp = np.linalg.svd(Pp)[2][-1]; Cp = Cp[:3] / Cp[3]
Xk = V3[k]

fig = plt.figure(figsize=(9, 6.5))
ax = fig.add_subplot(111, projection="3d")
for a, b in EDGES:
    ax.plot(*zip(V3[a], V3[b]), color=GREY, lw=1.0)
for Cc, lab in [(C, r"$\mathsf{C}$"), (Cp, r"$\mathsf{C}'$")]:
    ax.scatter(*Cc, s=45, color=BLUE)
    ax.text(*(Cc + np.array([0.3, 0.3, 0.3])), lab, color=BLUE, fontsize=12)
    ax.plot(*zip(Cc, Xk), color=ACCENT, lw=1.4)
ax.scatter(*Xk, s=60, color=ACCENT)
ax.text(*(Xk + np.array([0.2, 0.2, 0.4])), r"$\mathbf{X}$", color=ACCENT, fontsize=12)
ax.set_axis_off(); set_axes_equal(ax); ax.view_init(elev=16, azim=-64)
ax.set_title("the two optical rays meet at the world point")
plt.tight_layout(); plt.show()


Three points are now in play: the two camera centres and the world point. Three
points span a plane, and that plane organises everything.

::: {#def-epipolar-plane}
## Epipolar plane and baseline

The plane through $\mathsf{C}$, $\mathsf{C}'$ and $\mathbf{X}$ is the
**epipolar plane** of $\mathbf{X}$.

The line $\mathsf{C}\mathsf{C}'$ is the **baseline**. It is the same for every
world point, so every epipolar plane contains it: they form a pencil of planes
hinged on the baseline.
:::

::: {#def-epipoles}
## Epipoles

The baseline meets the first image plane in a point $\mathbf{e}$ and the second
in $\mathbf{e}'$: the **epipoles**. Equivalently, $\mathbf{e}$ is the image of
the second camera centre in the first view, and $\mathbf{e}'$ the image of the
first centre in the second view.
:::

::: {#def-epipolar-line}
## Epipolar lines

The epipolar plane cuts each image plane in a line: the **epipolar lines**
$\boldsymbol{\ell}$ and $\boldsymbol{\ell}'$. Every epipolar line passes through
the epipole of its image, since the baseline lies in every epipolar plane.
:::


In [ ]:
#| echo: false
def plane_of(P, depth):
    """Map pixel coordinates to a plane drawn `depth` cm in front of camera P."""
    Cc = np.linalg.svd(P)[2][-1]; Cc = Cc[:3] / Cc[3]
    Minv = np.linalg.inv(P[:, :3])
    axis = Minv @ np.array([W_IMG/2, H_IMG/2, 1.0])
    s = depth / np.linalg.norm(axis)
    return lambda uv: Cc + s * (Minv @ np.array([uv[0], uv[1], 1.0]))

DEPTH = 8.0
to3d, to3dp = plane_of(P, DEPTH), plane_of(Pp, DEPTH)
rect  = [(0, 0), (W_IMG, 0), (W_IMG, H_IMG), (0, H_IMG)]
quad  = np.array([to3d(c)  for c in rect])
quadp = np.array([to3dp(c) for c in rect])

x_k  = np.append(project(P,  Xk)[0], 1.0)
xp_k = np.append(project(Pp, Xk)[0], 1.0)

# the epipoles come straight from the definition: each centre seen by the other
# camera. No fundamental matrix is needed to draw this picture.
e_1 = P  @ np.append(Cp, 1.0); e_1 /= e_1[2]
e_2 = Pp @ np.append(C,  1.0); e_2 /= e_2[2]
l_1 = np.cross(e_1, x_k)              # an epipolar line joins the epipole
l_2 = np.cross(e_2, xp_k)             # to the image point

fig = plt.figure(figsize=(10, 7.5))
ax = fig.add_subplot(111, projection="3d")

for a, b in EDGES:
    ax.plot(*zip(V3[a], V3[b]), color=GREY, lw=1.1)

ax.add_collection3d(Poly3DCollection([np.array([C, Cp, Xk])], alpha=0.15,
                                     facecolor=ACCENT, edgecolor="none"))
for q in (quad, quadp):
    ax.add_collection3d(Poly3DCollection([q], alpha=0.13, facecolor=BLUE,
                                         edgecolor=BLUE, lw=0.9))

# the baseline, extended until it pierces both image planes: that is where
# the epipoles are
E1, E2 = to3d(e_1[:2]), to3dp(e_2[:2])
ax.plot(*zip(E1, E2), color=BLUE, lw=1.0, ls=(0, (4, 3)))
ax.plot(*zip(C, Cp), color=BLUE, lw=2.2)
for Cc in (C, Cp):
    ax.plot(*zip(Cc, Xk), color=ACCENT, lw=1.0, ls=(0, (5, 3)))

for l, f, lab in ((l_1, to3d, r"$\boldsymbol{\ell}$"),
                  (l_2, to3dp, r"$\boldsymbol{\ell}'$")):
    seg = clip_line_to_image(l, W_IMG, H_IMG)
    if seg is not None:
        p3 = np.array([f(p) for p in seg])
        ax.plot(*p3.T, color=ACCENT, lw=2.6)
        ax.text(*(p3[0] + np.array([0, 0, 0.7])), lab, color=ACCENT, fontsize=14)

for uv, f, lab, col in ((x_k[:2],  to3d,  r"$\mathbf{x}$",  ACCENT),
                        (xp_k[:2], to3dp, r"$\mathbf{x}'$", ACCENT),
                        (e_1[:2],  to3d,  r"$\mathbf{e}$",  BLUE),
                        (e_2[:2],  to3dp, r"$\mathbf{e}'$", BLUE)):
    Q = f(uv)
    ax.scatter(*Q, s=38, color=col, depthshade=False, zorder=6)
    ax.text(*(Q + np.array([0.35, 0.35, 0.35])), lab, color=col, fontsize=13)

for Cc, lab in ((C, r"$\mathsf{C}$"), (Cp, r"$\mathsf{C}'$")):
    ax.scatter(*Cc, s=48, color=BLUE, depthshade=False, zorder=6)
    ax.text(*(Cc + np.array([0.5, 0.5, 0.5])), lab, color=BLUE, fontsize=14)
ax.scatter(*Xk, s=58, color=ACCENT, depthshade=False, zorder=6)
ax.text(*(Xk + np.array([0.3, 0.3, 0.8])), r"$\mathbf{X}$", color=ACCENT, fontsize=14)

pts = np.vstack([np.array(list(V3.values())), quad, quadp, [C], [Cp], [E1], [E2]])
lo, hi = pts.min(0) - 0.8, pts.max(0) + 0.8
ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_zlim(lo[2], hi[2])
ax.set_box_aspect(hi - lo)
ax.view_init(elev=12, azim=-72)
ax.set_axis_off()
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()


Fix $\mathbf{x}$ in the first image. Its
ray lies in the epipolar plane. The
world point, wherever along the ray it happens to be, and therefore its image in
the second view. So $\mathbf{x}'$ is **not free**. It has to lie on the line
where the epipolar plane cuts the second image plane.

We can write that line down without knowing $\mathbf{X}$ at all. Two points of the first camera's ray are enough to determine it: the centre
$\mathsf{C}$, and any other point of the ray, for instance
$\mathsf{P}^{+}\mathbf{x}$, where $\mathsf{P}^{+}$ is the pseudo-inverse. That
second point is an ordinary finite point of the ray — the pseudo-inverse returns
some $\mathbf{Y}$ with $\mathsf{P}\mathbf{Y} = \mathbf{x}$, and which one it
returns does not matter, since any two distinct points of the ray span the same
line. Project both with $\mathsf{P}'$ and join them:

$$\boldsymbol{\ell}' \;=\; (\mathsf{P}'\mathsf{C}) \times
(\mathsf{P}'\mathsf{P}^{+}\mathbf{x}) \;=\; \mathbf{e}' \times
(\mathsf{P}'\mathsf{P}^{+}\mathbf{x}) \;=\;
\underbrace{[\mathbf{e}']_\times \mathsf{P}'\mathsf{P}^{+}}_{\textstyle \mathsf{F}}\;
\mathbf{x}.$$

There is $\mathsf{F}$ again, and this time we can see what it does: it takes a
point in the first image and returns a **line** in the second. Let us check that
the correspondences really do lie on the lines it predicts.


In [ ]:
Pplus = np.linalg.pinv(P)
e_p = Pp @ np.append(C, 1.0)                 # the second epipole
F_geo = skew(e_p) @ Pp @ Pplus
F_geo = F_geo / np.linalg.norm(F_geo)

X_h = h(xs); Xp_h = h(xps)
lines = (F_geo @ X_h.T).T                    # one epipolar line per point
dist = np.abs(np.sum(Xp_h * lines, axis=1)) / np.linalg.norm(lines[:, :2], axis=1)

for name, dd in zip(IDS, dist):
    print(f"  {name:3s}  distance of x' from its epipolar line: {dd:.2e} px")


Zero to machine precision, as it must be on exact data.

Now an important property. The map
$\mathbf{x} \mapsto \mathsf{F}\mathbf{x}$ goes from $\mathbb{P}^2$ to
$\mathbb{P}^{2*}$, points to lines. **It is not injective.** Take any two points
of the first image that lie on a common line through the epipole: they belong to
the same epipolar plane, so they must produce the same epipolar line.


In [ ]:
e = np.linalg.svd(F_geo)[2][-1]; e = e / e[2]     # first epipole
x_a = X_h[IDS.index("X9")]
x_b = e + 0.35 * (x_a - e)                        # another point of the same ray

l_a = F_geo @ x_a; l_a /= np.linalg.norm(l_a[:2])
l_b = F_geo @ x_b; l_b /= np.linalg.norm(l_b[:2])

print("x_a :", np.round(x_a[:2], 1))
print("x_b :", np.round(x_b[:2], 1), "  (a different point of the first image)")
print("their epipolar lines differ by:", f"{np.abs(np.abs(l_a) - np.abs(l_b)).max():.2e}")


Two different points, one line. A map that collapses a whole pencil of points
onto a single image cannot be invertible, and for a linear map that means

$$\operatorname{rank} \mathsf{F} = 2, \qquad \det \mathsf{F} = 0 .$$

We could have seen this coming from the construction: $\mathsf{F} =
[\mathbf{e}']_\times \mathsf{P}'\mathsf{P}^{+}$ contains a skew-symmetric
$3\times 3$ factor, and those always have rank two.

The rank deficiency also tells us what the kernel is. $\mathsf{F}\mathbf{e} =
\mathbf{0}$: the epipole is the one point of the first image that has no
epipolar line, because its ray *is* the baseline and its epipolar plane is not
determined. So the epipoles come for free from $\mathsf{F}$ alone, as null
vectors.


In [ ]:
def epipoles(F):
    """e spans the right null space of F, e' the right null space of F^T."""
    e  = np.linalg.svd(F)[2][-1];   e  = e / e[2]
    ep = np.linalg.svd(F.T)[2][-1]; ep = ep / ep[2]
    return e, ep

e, ep = epipoles(F_geo)
print("e  =", np.round(e[:2], 0), "   e' =", np.round(ep[:2], 0))
print(f"the frame is {W_IMG} x {H_IMG}, so both fall outside the picture,")
print("to the left: the epipolar lines converge somewhere off-frame.")
print()
print("check: e is the image of the other camera centre")
q = P @ np.append(Cp, 1.0)
print("  P C' =", np.round(q[:2] / q[2], 0))


## 3. Computing $\mathsf{F}$ from correspondences

Everything so far assumed we knew the cameras. In practice we usually do not:
we have two photographs and a set of matched points, and we would like the
epipolar geometry out of them.

The whole algorithm is four lines.

1. Each correspondence gives one linear equation in the nine entries of
   $\mathsf{F}$.
2. Stack eight or more of them and take the null vector of the resulting matrix.
3. Do all of this in normalized coordinates, or the answer will be numerical
   noise.
4. Force the result to have rank two.

Step one first. Expanding $\mathbf{x}'^\top \mathsf{F} \mathbf{x} = 0$ with
$\mathbf{x} = (u, v, 1)$ and $\mathbf{x}' = (u', v', 1)$ gives

$$
\begin{bmatrix} u'u & u'v & u' & v'u & v'v & v' & u & v & 1\end{bmatrix}
\operatorname{vec}\mathsf{F} = 0,
$$

which is the Kronecker product $(\mathbf{x}' \otimes \mathbf{x})^\top
\operatorname{vec}\mathsf{F} = 0$: bilinear in the points, linear in the
unknowns.

How many do we need? $\mathsf{F}$ has nine entries, but it is defined only up to
scale — multiplying it by any non-zero constant leaves the constraint unchanged
— and it must have rank two. That is nine minus one minus one:

$$\text{7 degrees of freedom.}$$

Seven correspondences ought to be enough, and they are: that is a different
notebook, because the rank condition is cubic and the solution is not unique.
If we drop the rank condition and treat $\mathsf{F}$ as an arbitrary matrix
defined up to scale, we need **eight**, and the problem becomes linear. Hence
the name.


In [ ]:
def design_matrix(x, xp):
    """One row per correspondence: the Kronecker product x' (x) x."""
    u,  v  = x[:, 0],  x[:, 1]
    up, vp = xp[:, 0], xp[:, 1]
    return np.column_stack([up*u, up*v, up,
                            vp*u, vp*v, vp,
                            u,    v,    np.ones(len(x))])

A = design_matrix(X_h, Xp_h)
print("design matrix:", A.shape)
print("column magnitudes:", np.round(np.abs(A).max(axis=0), 1))


Look at that last line before going on. The entries of $\mathsf{A}$ span six
orders of magnitude: the $u'u$ column is of order $10^6$, the $u$ column of
order $10^3$, the last column is exactly $1$. We are about to ask for the
smallest singular vector of this matrix, and a matrix whose columns live on
wildly different scales has a condition number to match. The answer will be
dominated by rounding.

Hartley's fix is to change coordinates before solving: translate each set of
points so its centroid is at the origin, then scale so the average distance from
the origin is $\sqrt2$. Both are similarities, and a similarity of the image is
just a different choice of pixel units — the geometry does not care.


In [ ]:
#| echo: false
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
pos = {"p2": (1.4, 4.6), "p2s": (8.6, 4.6), "n2": (1.4, 1.2), "n2s": (8.6, 1.2)}
lab = {"p2": r"$\mathbb{P}^2$", "p2s": r"$\mathbb{P}^{2*}$",
       "n2": r"$\widehat{\mathbb{P}}^2$", "n2s": r"$\widehat{\mathbb{P}}^{2*}$"}
for k, (xx, yy) in pos.items():
    ax.text(xx, yy, lab[k], fontsize=17, ha="center", va="center")
arrows = [("p2", "p2s", r"$\mathsf{F}$", 0.35),
          ("n2", "n2s", r"$\widehat{\mathsf{F}}$", -0.55),
          ("p2", "n2", r"$\mathsf{T}$", 0.0),
          ("p2s", "n2s", r"$\mathsf{T}'^{-\top}$", 0.0)]
for a, b, t, off in arrows:
    (x0, y0), (x1, y1) = pos[a], pos[b]
    dx, dy = x1 - x0, y1 - y0
    n = np.hypot(dx, dy); ux, uy = dx / n, dy / n
    ax.annotate("", xy=(x1 - 0.7*ux, y1 - 0.45*uy), xytext=(x0 + 0.7*ux, y0 + 0.45*uy),
                arrowprops=dict(arrowstyle="-|>", color=GREY, lw=1.3))
    ax.text((x0+x1)/2 + (0 if dx else 0.45), (y0+y1)/2 + off,
            t, fontsize=14, color=ACCENT, ha="center", va="center")
ax.set_title("solve downstairs, then carry the answer back up", fontsize=11)
plt.tight_layout(); plt.show()


The diagram is the whole idea: we cannot solve well upstairs, so we push the
points down with $\mathsf{T}$ and $\mathsf{T}'$, solve there for
$\widehat{\mathsf{F}}$, and carry the result back with

$$\mathsf{F} = \mathsf{T}'^\top \widehat{\mathsf{F}}\, \mathsf{T}.$$

The transpose on $\mathsf{T}'$ is not a typo — lines transform contravariantly
to points, which is exactly what the right-hand arrow of the diagram says.


In [ ]:
def normalize_points(x):
    """Centroid to the origin, mean distance to the origin equal to sqrt(2)."""
    c = x[:, :2].mean(axis=0)
    d = np.linalg.norm(x[:, :2] - c, axis=1).mean()
    s = np.sqrt(2.0) / d
    T = np.array([[s, 0, -s*c[0]],
                  [0, s, -s*c[1]],
                  [0, 0,  1.0]])
    return (T @ x.T).T, T

xn, T = normalize_points(X_h)
print("before:  mean distance from the centroid =",
      f"{np.linalg.norm(X_h[:, :2] - X_h[:, :2].mean(0), axis=1).mean():8.1f} px")
print("after :  mean distance from the centroid =",
      f"{np.linalg.norm(xn[:, :2] - xn[:, :2].mean(0), axis=1).mean():8.4f}",
      "  (= sqrt(2))")
# the ninth singular value is the solution direction, so compare the eight
# that carry information: s1 / s8 is the conditioning that actually bites
s_raw = np.linalg.svd(design_matrix(X_h, Xp_h))[1]
s_nrm = np.linalg.svd(design_matrix(xn, normalize_points(Xp_h)[0]))[1]
print(f"spread of the informative singular values, s1/s8:")
print(f"   in pixels     {s_raw[0]/s_raw[7]:.1e}")
print(f"   normalized    {s_nrm[0]/s_nrm[7]:.1e}")


Six orders of magnitude of conditioning, recovered by a change of units.

One step remains. The null vector of $\mathsf{A}$ is some $3 \times 3$ matrix,
and on noisy data it will have rank three: a matrix that is *nearly* a
fundamental matrix but not quite. We replace it by the closest rank-two matrix
in the Frobenius norm, which the singular value decomposition hands us for free —
compute $\widehat{\mathsf{F}} = \mathsf{U}\operatorname{diag}(\sigma_1, \sigma_2,
\sigma_3)\mathsf{V}^\top$ and set $\sigma_3$ to zero.


In [ ]:
def eight_point(x, xp, enforce_rank2=True):
    """Estimate F from n >= 8 correspondences, in homogeneous pixel coordinates."""
    xn,  T  = normalize_points(x)
    xpn, Tp = normalize_points(xp)

    A = design_matrix(xn, xpn)
    Fh = np.linalg.svd(A)[2][-1].reshape(3, 3)

    if enforce_rank2:
        U, s, Vt = np.linalg.svd(Fh)
        s[-1] = 0.0
        Fh = U @ np.diag(s) @ Vt

    F = Tp.T @ Fh @ T
    return F / np.linalg.norm(F)


def epipolar_distance(F, x, xp):
    """Symmetric point-to-epipolar-line distance, in pixels."""
    l  = (F @ x.T).T
    lp = (F.T @ xp.T).T
    d1 = np.abs(np.sum(xp * l,  axis=1)) / np.linalg.norm(l[:,  :2], axis=1)
    d2 = np.abs(np.sum(x  * lp, axis=1)) / np.linalg.norm(lp[:, :2], axis=1)
    return (d1 + d2) / 2


Time to try it on data where we know the answer. The synthetic projections are
exact, and we already have the true $\mathsf{F}$ from the cameras, so the
estimate has nowhere to hide.


In [ ]:
F_est = eight_point(X_h, Xp_h)
sgn = np.sign((F_est * F_geo).sum())

print("residuals on the synthetic data:",
      f"{epipolar_distance(F_est, X_h, Xp_h).max():.2e} px")
print("largest entrywise difference from the true F:",
      f"{np.abs(sgn*F_est - F_geo).max():.2e}")


So you were able to recover the fundamental matrix up to machine precision. In practice, however we have noise, and to improve the statistical efficiecny of our estimation, instead of considering just 8 points, we can increase the numeber of matches. We just used ten
correspondences, and the extra two do no harm, the system becomes
overdetermined and the null vector becomes a least-squares fit, which is exactly
what we want when the data is noisy. More points, better estimate.

And a warning that costs nothing to state and a lot to discover: eight points in
a *bad configuration* are worse than useless. Here is the same estimate run on
every subset of eight, sorted by how far it lands from the truth.


In [ ]:
import itertools
scores = []
for c in itertools.combinations(range(len(IDS)), 8):
    c = list(c)
    Fc = eight_point(X_h[c], Xp_h[c])
    err = np.abs(np.sign((Fc*F_geo).sum())*Fc - F_geo).max()
    scores.append((err, tuple(IDS[i] for i in c)))
scores.sort()
print("best  subset:", scores[0][1],  f"  error {scores[0][0]:.1e}")
print("worst subset:", scores[-1][1], f"  error {scores[-1][0]:.1e}")


In [ ]:
#| echo: false
def subset_diag(c):
    x, xp = X_h[list(c)], Xp_h[list(c)]
    xn, T = normalize_points(x); xpn, Tp = normalize_points(xp)
    A = design_matrix(xn, xpn)
    S = np.linalg.svd(A)[1]
    Fh = np.linalg.svd(A)[2][-1].reshape(3, 3)
    return S[-1] / S[0], np.linalg.svd(Fh)[1][-1] / np.linalg.svd(Fh)[1][0]

best_ids, worst_ids = scores[0][1], scores[-1][1]
fig = plt.figure(figsize=(13, 5.6))
for j, (sel, tag, err) in enumerate([(best_ids, "a subset that works", scores[0][0]),
                                     (worst_ids, "a subset that does not", scores[-1][0])]):
    idx = [IDS.index(s) for s in sel]
    s8, s3 = subset_diag(idx)
    ax = fig.add_subplot(1, 2, j + 1, projection="3d")
    for a, b in EDGES:
        ax.plot(*zip(V3[a], V3[b]), color="0.78", lw=0.8)
    for k, Q in V3.items():
        on = k in sel
        ax.scatter(*Q, s=42 if on else 14,
                   color=ACCENT if on else "0.7", depthshade=False, zorder=5)
    d = model["dimensions_cm"]
    ax.set_box_aspect((d["long_side"], d["depth"], d["total_height"]))
    ax.view_init(elev=18, azim=-62); ax.set_axis_off()
    ax.set_title(f"{tag}\n{' '.join(sel)}\n"
                 f"error {err:.0e}    "
                 r"$\sigma_8/\sigma_1$ = " f"{s8:.0e}    "
                 r"$\sigma_3/\sigma_1$ of $\widehat{\mathsf{F}}$ = " f"{s3:.2f}",
                 fontsize=10)
plt.tight_layout(); plt.show()


On exact data the estimate is either perfect or badly wrong, with nothing in
between — and the difference is visible before solving anything. In the bad
subsets the design matrix has **two** vanishing singular values instead of one:
the eight correspondences do not determine a single $\mathsf{F}$ but a whole
pencil of them, and the algorithm returns an arbitrary member of that pencil.

A second symptom appears downstream. In the good subsets the raw
$\widehat{\mathsf{F}}$ already has rank two to machine precision, so enforcing
the constraint changes nothing; in the bad ones its third singular value is a
sizeable fraction of the first, and the projection has to move a long way.

Eight points are enough only when they are in general position. Which
configurations fail, and why, is a notebook of its own.


### On images

Let's Now  use real correspondences
clicked by hand on the two photographs — ten of them, the six house corners and
the four corners of the door. Their mean distance from the true epipolar lines
is about 1.7 pixels, which is what careful annotation looks like.


In [ ]:
ann = load_annotation("two_view_matches")
xr  = h(np.array([m["x"]  for m in ann["matches"]]))
xpr = h(np.array([m["xp"] for m in ann["matches"]]))
rid = [m["id"] for m in ann["matches"]]

d_true = epipolar_distance(F_geo, xr, xpr)
for name, dd in zip(rid, d_true):
    print(f"  {name:3s}  {dd:5.2f} px from the true epipolar line")
print(f"\n  mean {d_true.mean():.2f}   median {np.median(d_true):.2f}   max {d_true.max():.2f}")


In [ ]:
#| echo: false
fig, axes = plt.subplots(1, 2, figsize=(13, 8))
for ax, im, pts, prime, ttl in [
        (axes[0], I,  xr,  False, ann["images"]["view"]),
        (axes[1], Ip, xpr, True,  ann["images"]["view_prime"])]:
    ax.imshow(im)
    ax.scatter(pts[:, 0], pts[:, 1], s=60, facecolors="none",
               edgecolors=ACCENT, lw=2, zorder=5)
    for name, q in zip(rid, pts):
        lab = rf"$\mathbf{{x}}'_{{\mathrm{{{name}}}}}$" if prime \
              else rf"$\mathbf{{x}}_{{\mathrm{{{name}}}}}$"
        ax.text(q[0] + 16, q[1] - 16, lab, fontsize=9, color=BLUE,
                bbox=dict(fc="white", alpha=0.7, ec="none", pad=1))
    ax.set_xlim(pts[:, 0].min() - 150, pts[:, 0].max() + 150)
    ax.set_ylim(pts[:, 1].max() + 150, pts[:, 1].min() - 150)
    ax.set_title(ttl, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()


Now estimate $\mathsf{F}$ from these, twice: once letting the null vector stand
as it comes out of the linear system, once forcing rank two. The difference is
invisible in the numbers and unmistakable in the picture.


In [ ]:
F_free = eight_point(xr, xpr, enforce_rank2=False)
F_rank2 = eight_point(xr, xpr, enforce_rank2=True)

for tag, F in [("no rank constraint", F_free), ("rank two enforced", F_rank2)]:
    print(f"{tag:20s}  numerical rank {np.linalg.matrix_rank(F)}   "
          f"residuals: mean {epipolar_distance(F, xr, xpr).mean():5.2f} px")


In [ ]:
#| echo: false
# frame both panels identically, wide enough to hold the image and the
# place where the lines are heading
allpts = np.vstack([pairwise_intersections((F @ xr.T).T) for F in (F_free, F_rank2)])
ctr = np.median(allpts, axis=0)
x0 = min(0, ctr[0]) - 300; x1 = max(W_IMG, ctr[0]) + 300
y0 = min(700, ctr[1]) - 300; y1 = max(H_IMG, ctr[1]) + 300

fig, axes = plt.subplots(1, 2, figsize=(13, 8))
for ax, F, ttl in [(axes[0], F_free,  "no rank constraint"),
                   (axes[1], F_rank2, "rank two enforced")]:
    ax.imshow(Ip)
    L = (F @ xr.T).T
    for l in L:
        seg = clip_line_to_image(l, W_IMG, H_IMG)
        if seg is None:
            continue
        # extend the drawn segment towards the meeting point
        d = seg[1] - seg[0]; d = d / np.linalg.norm(d)
        far = np.vstack([seg[0] - 2600*d, seg[1] + 2600*d])
        ax.plot(far[:, 0], far[:, 1], color=ACCENT, lw=0.9, alpha=0.85)
    ax.scatter(xpr[:, 0], xpr[:, 1], s=45, facecolors="none",
               edgecolors=BLUE, lw=1.6, zorder=5)
    pts = pairwise_intersections(L)
    c = np.median(pts, axis=0)
    sp = np.median(np.linalg.norm(pts - c, axis=1))
    ax.scatter(pts[:, 0], pts[:, 1], s=14, color=BLUE, alpha=0.6, zorder=6)
    ax.set_title(f"{ttl}\nmedian spread of the 45 intersections: {sp:.0f} px",
                 fontsize=10)
    ax.set_xlim(x0, x1); ax.set_ylim(y1, y0); ax.set_aspect("equal")
    ax.axis("off")
plt.tight_layout(); plt.show()


On the left the epipolar lines almost meet: the
small blue dots are the 45 pairwise intersections, scattered over a couple of
hundred pixels. There is no epipole, because a rank-three matrix has no null
vector, and a family of lines with no common point is not an epipolar pencil.

On the right they meet exactly, at a single point. Setting one singular value to
zero is what turns a matrix that fits the data into a matrix that describes the epiplar
geometry.

Please note also that Transposing the constraint
gives $\mathbf{x}^\top \mathsf{F}^\top \mathbf{x}' = 0$: the same matrix, read
backwards, maps points of the second image to lines in the first.


In [ ]:
#| echo: false
L = (F_rank2.T @ xpr.T).T
pts = pairwise_intersections(L); ctr = np.median(pts, axis=0)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(I)
for l in L:
    seg = clip_line_to_image(l, W_IMG, H_IMG)
    if seg is None:
        continue
    d = seg[1] - seg[0]; d = d / np.linalg.norm(d)
    far = np.vstack([seg[0] - 2600*d, seg[1] + 2600*d])
    ax.plot(far[:, 0], far[:, 1], color=ACCENT, lw=0.9)
ax.scatter(xr[:, 0], xr[:, 1], s=45, facecolors="none",
           edgecolors=BLUE, lw=1.6, zorder=5)
ax.scatter(*ctr, s=60, color=BLUE, zorder=6)
ax.text(ctr[0] + 40, ctr[1] - 40, r"$\mathbf{e}$", color=BLUE, fontsize=13)
ax.set_title(r"$\mathsf{F}^\top$ carries the second view's points"
             "\n" r"to epipolar lines in the first", fontsize=11)
ax.set_xlim(min(0, ctr[0]) - 250, max(W_IMG, ctr[0]) + 250)
ax.set_ylim(max(H_IMG, ctr[1]) + 250, min(700, ctr[1]) - 250)
ax.set_aspect("equal"); ax.axis("off")
plt.tight_layout(); plt.show()


## Closing the circle

We began with $\det\mathsf{L} = 0$ and argued that it had to be a bilinear form.
Let us collect what $\mathsf{F}$ turned out to be.

Expanding the determinant by cofactors, each entry of $\mathsf{F}$ is a
$4\times4$ minor built from two rows of $\mathsf{P}$ and two rows of
$\mathsf{P}'$:

$$\mathsf{F}_{ji} \;=\; (-1)^{i+j}\,
\det\!\begin{bmatrix} \widetilde{\mathsf{P}}_i \\ \widetilde{\mathsf{P}}'_j
\end{bmatrix},$$

where $\widetilde{\mathsf{P}}_i$ is $\mathsf{P}$ with row $i$ removed. So
$\mathsf{F}$ is computable from the cameras alone, with no geometry and no
correspondences — and the rank-two property, which we had to *impose* on the
estimate, comes out of this construction on its own.


In [ ]:
def F_from_cameras(P, Pp):
    """Each entry of F is a 4x4 minor of the two stacked camera matrices."""
    F = np.zeros((3, 3))
    for i in range(3):
        for j in range(3):
            rows = ([P[k]  for k in range(3) if k != i] +
                    [Pp[k] for k in range(3) if k != j])
            F[j, i] = (-1)**(i+j) * np.linalg.det(np.vstack(rows))
    return F / np.linalg.norm(F)

F_min = F_from_cameras(P, Pp)
print("rank of F built from the minors:", np.linalg.matrix_rank(F_min),
      "  (nobody asked for it)")
print("agreement with the geometric construction:",
      f"{np.abs(np.sign((F_min*F_geo).sum())*F_min - F_geo).max():.2e}")

# The two expressions agree up to one global scale factor. On corresponding
# pairs both vanish, so we test on pairs that do NOT correspond, where the
# numbers are large and the ratio has something to say.
print("\n   pair          det L        x'^T F x       ratio")
for i, j in [(0, 5), (2, 7), (3, 9), (6, 1)]:
    dL = np.linalg.det(L_matrix(P, Pp, X_h[i], Xp_h[j]))
    q  = Xp_h[j] @ F_min @ X_h[i]
    print(f"   {IDS[i]:>3s}/{IDS[j]:<3s}  {dL:12.4e}  {q:12.4e}  {dL/q:11.4f}")
print("\n   one constant ratio, as a bilinear form defined up to scale must give.")


### And backwards: cameras from $\mathsf{F}$

If $\mathsf{F}$ comes from the cameras, can we go the other way? Only partly, and
the obstruction is worth seeing because it is the reason uncalibrated
reconstruction is *projective* reconstruction.

Replace the pair $(\mathsf{P}, \mathsf{P}')$ by $(\mathsf{P}\mathsf{H}^{-1},
\mathsf{P}'\mathsf{H}^{-1})$ for any invertible $4\times4$ matrix $\mathsf{H}$,
and move every world point to $\mathsf{H}\mathbf{X}$. Nothing in the images
changes, since $\mathsf{P}\mathsf{H}^{-1}\mathsf{H}\mathbf{X} =
\mathsf{P}\mathbf{X}$. So $\mathsf{F}$ cannot possibly distinguish the two
pairs: it determines the cameras **only up to a projective transformation of
space**.


Within that freedom we may as well make a convenient choice. Sending the first
camera to $[\mathsf{I} \mid \mathbf{0}]$ fixes the world frame to the first
camera's own, and then

$$\mathsf{P} = [\mathsf{I} \mid \mathbf{0}], \qquad
  \mathsf{P}' = [\,[\mathbf{e}']_\times \mathsf{F} \mid \mathbf{e}'\,]$$

is a pair that produces exactly this $\mathsf{F}$.


In [ ]:
F_c = F_rank2
_, ep_c = epipoles(F_c)

P_can  = np.hstack([np.eye(3), np.zeros((3, 1))])
Pp_can = np.hstack([skew(ep_c) @ F_c, ep_c[:, None]])

F_back = F_from_cameras(P_can, Pp_can)
print("F recovered from the canonical pair, compared with the F we started from:",
      f"{np.abs(np.sign((F_back*F_c).sum())*F_back - F_c).max():.2e}")


### Questions to leave open

**We force the rank down to two — what stops it from falling to one?** Nothing,
in the algorithm as written. A rank-one $\mathsf{F}$ is degenerate in a way that
the SVD step will happily produce if the second singular value is also small.
When does that happen, and would you notice?

**Forcing rank two made the residuals worse. Should it not have made them
better?** Compare the numbers above: the unconstrained estimate fits the ten
points more closely than the rank-two one, and both fit them more closely than
the *true* $\mathsf{F}$. What is being minimized, and is it what we care about?

**The epipoles estimated from the photographs are nowhere near the true ones.**
Compute them and see. Ten points, annotated to two pixels, and a quantity that
moves by hundreds — which part of the geometry is so badly conditioned, and
what would you do about it?

**What happens when the camera moves straight ahead?** The baseline points along
the optical axis, so the epipole lands inside the picture and the epipolar lines
radiate out from it. Where exactly, and what does the pencil look like?

**Is every rank-two $3\times3$ matrix a fundamental matrix?** For calibrated
cameras the analogous object, the essential matrix, needs two equal singular
values and has only five degrees of freedom. For $\mathsf{F}$ the answer is
different, and it explains why the seven-point algorithm has the shape it has.

**For what motion is $\mathsf{F}$ skew-symmetric?** Then every epipolar line
corresponds to itself, and the two pictures are related in a way you can spot by
eye.

**Does $\mathsf{F}$ still exist if the two photographs were taken at different
zoom?** The answer is the reason one works with $\mathsf{F}$ at all.


## Further reading

- Longuet-Higgins, H. C. "A computer algorithm for reconstructing a scene from two projections", *Nature* 293, 1981. The original eight-point algorithm.
- Hartley, R. "In defense of the eight-point algorithm", *IEEE TPAMI* 19(6), 1997. Why normalizing the coordinates matters as much as it does.
- Hartley, R. and Zisserman, A. *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press, 2004. Chapter 9 for the epipolar geometry, Chapter 11 for the computation of $\mathsf{F}$, and Result 17.1 for the minors.
- Fusiello, A. *Visione Computazionale: tecniche di ricostruzione tridimensionale*, Franco Angeli, 2018. Chapter 5.

---

**Luca Magri** — Computer Vision Dojo
Code MIT · text and figures CC BY-NC-ND 4.0
<https://magrilu.github.io/cv-dojo/>
